# Qwen3.5-9B QLoRA on an M5 Pro Mac with 48 GB (MLX)

This notebook fine-tunes the 4-bit `mlx-community/Qwen3.5-9B-4bit` checkpoint for rubric grading on a 48 GB M5 Pro Mac. It uses MLX/Metal, masks the prompt so loss is calculated only on the assistant JSON answer, and verifies the Mac's unified-memory capacity.

Requirements: Apple Silicon, macOS 14 or newer, Python 3.11 or 3.12, 48 GB unified memory for the target profile (a conservative 32 GB fallback is included), roughly 15 GB free disk space, and `training_data.csv` beside this notebook. Allow extra disk space for retained adapter checkpoints. Close memory-heavy applications before training. The initial model download is approximately 6 GB; runtime memory also includes activations, adapters, optimizer state, and Metal allocations.

Run the cells in order. Training is gated behind `RUN_TRAINING = False`; enable it after reviewing the generated configuration.


## 1. Install the native Apple Silicon stack

Use a native arm64 Python environment, not a Rosetta/x86 environment. Restart the kernel after this cell if Jupyter requests it.

In [ ]:
%pip install -U "mlx-lm[train]>=0.30.7" pandas pyyaml transformers ipywidgets

## 2. Hardware check and memory-aware profile

The Qwen3.5-9B 4-bit checkpoint is approximately 6 GB on disk. The 48 GB M5 Pro starting profile trains adapters in the final 16 transformer layers at rank 16, with micro-batch size 2 and gradient checkpointing enabled to leave memory headroom. Four micro-batches give an effective batch size of 8. These settings need a local training run to confirm peak memory and throughput.

A 32 GB fallback uses 8 layers, rank 8, and micro-batch size 1 with eight accumulation steps. Larger-memory Macs retain the target 48 GB profile.


In [ ]:
import ast
import csv
import json
import math
import platform
import re
import subprocess
import sys
from pathlib import Path

import mlx.core as mx
import pandas as pd
import yaml

if platform.system() != "Darwin" or platform.machine() != "arm64":
    raise RuntimeError("This notebook must run in a native arm64 Python environment on Apple Silicon.")
if not mx.metal.is_available():
    raise RuntimeError("MLX cannot access Metal. Check the Python architecture and macOS installation.")

memory_bytes = int(subprocess.check_output(["sysctl", "-n", "hw.memsize"], text=True).strip())
MEMORY_GB = round(memory_bytes / 1024**3)

if MEMORY_GB < 32:
    raise MemoryError(
        f"Detected {MEMORY_GB} GB. This Qwen3.5-9B notebook requires at least 32 GB "
        "for its conservative fallback; the target M5 Pro profile uses 48 GB."
    )
elif MEMORY_GB < 48:
    PROFILE = {
        "name": "Qwen3.5-9B / 32 GB conservative", "num_layers": 8, "rank": 8,
        "batch_size": 1, "grad_accumulation_steps": 8, "grad_checkpoint": True,
        "clear_cache_threshold": 0,
    }
else:
    PROFILE = {
        "name": "Qwen3.5-9B / M5 Pro 48 GB", "num_layers": 16, "rank": 16,
        "batch_size": 2, "grad_accumulation_steps": 4, "grad_checkpoint": True,
        "clear_cache_threshold": 1024**3,
    }

print(f"Python: {sys.version.split()[0]} ({platform.machine()})")
print(f"MLX: {mx.__version__}; Metal available: {mx.metal.is_available()}")
print(f"Unified memory: {MEMORY_GB} GB")
print("Selected profile:", PROFILE)

## 3. Experiment configuration

Train Qwen3.5-9B for approximately three epochs, using a learning rate of `2e-5`, LoRA dropout of 0.05, and a 512-token sequence cap. The iteration count is calculated from the training split. MLX-LM uses full micro-batches and drops a final incomplete batch; the count is rounded up to finish the last gradient-accumulation update. Review validation loss before increasing epochs or learning rate.


In [ ]:
MODEL_ID = "mlx-community/Qwen3.5-9B-4bit"
DATASET_PATH = Path("./training_data.csv")
DATA_DIR = Path("./mlx-qwen35-9b-data")
OUTPUT_DIR = Path("./qwen35-9b-grading-mlx-lora")
CONFIG_PATH = Path("./qwen35-9b-m5-lora.yaml")

SEED = 42
NUM_EPOCHS = 3
MAX_SEQ_LENGTH = 512
BATCH_SIZE = PROFILE["batch_size"]
GRAD_ACCUMULATION_STEPS = PROFILE["grad_accumulation_steps"]
LEARNING_RATE = 2e-5
LORA_DROPOUT = 0.05

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Place training_data.csv at {DATASET_PATH.resolve()}")

## 4. Load and validate the grading data

In [ ]:
df = pd.read_csv(DATASET_PATH)
required = ["question", "desired_answer", "student_answer", "criteria[]", "criteria_score[]"]
missing = [column for column in required if column not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

def parse_array(value):
    """Parse JSON/Python lists and the dataset's unquoted bracket format."""
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    text = str(value).strip()
    for _ in range(2):
        try:
            parsed = json.loads(text)
        except json.JSONDecodeError:
            break
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, str):
            text = parsed.strip()
            continue
        break
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, str):
            text = parsed.strip()
    except (ValueError, SyntaxError):
        pass
    if not (text.startswith("[") and text.endswith("]")):
        raise ValueError(f"Expected a bracketed list, got {value!r}")
    inner = text[1:-1].strip()
    if not inner:
        return []
    if re.search(r",(?=Answer\s)", inner):
        return [item.strip() for item in re.split(r",(?=Answer\s)", inner)]
    return [item.strip() for item in next(csv.reader([inner], skipinitialspace=True))]

def parse_criteria(value, expected_count):
    items = parse_array(value)
    if len(items) == expected_count:
        return items
    text = str(value).strip()
    for _ in range(2):
        try:
            decoded = json.loads(text)
        except json.JSONDecodeError:
            break
        if isinstance(decoded, str):
            text = decoded.strip()
        else:
            break
    inner = text[1:-1].strip() if text.startswith("[") and text.endswith("]") else text
    items = [inner] if expected_count == 1 else re.split(r",(?=Answer\s)", inner)
    items = [item.strip() for item in items]
    if len(items) != expected_count:
        raise ValueError(f"Parsed {len(items)} criteria but expected {expected_count}: {value!r}")
    return items

df["criteria_score"] = df["criteria_score[]"].apply(lambda v: [int(x) for x in parse_array(v)])
df["criteria"] = df.apply(lambda r: parse_criteria(r["criteria[]"], len(r["criteria_score"])), axis=1)

bad_rows = []
for index, row in df.iterrows():
    if not row["criteria"] or len(row["criteria"]) != len(row["criteria_score"]):
        bad_rows.append((index, "criterion/label length mismatch"))
    elif any(int(score) not in (0, 1) for score in row["criteria_score"]):
        bad_rows.append((index, "non-binary label"))
if bad_rows:
    raise ValueError(f"Invalid rows (first 10): {bad_rows[:10]}")
print(f"Validated {len(df):,} rows")
df[["criteria", "criteria_score"]].head()

## 5. Build assistant-only chat examples and leakage-safe deterministic splits

The final chat message is the completion. `mask_prompt: true` in the MLX configuration excludes the system and user tokens from loss. Grading totals and maximum scores never enter the prompt. Rows sharing an assignment/question ID stay in the same split so validation and test results measure generalization to unseen questions.

In [ ]:
SYSTEM_PROMPT = """You are a strict grading assistant.

Evaluate whether a student's answer fulfills each grading criterion independently.
For every criterion, output 1 if it is fulfilled and 0 if it is not.
Return ONLY a valid JSON array containing exactly one 0 or 1 per criterion, in the original order.
Do not include explanations, markdown, or any text outside the JSON array."""

def make_example(row):
    criteria_text = "\n".join(f"{i + 1}. {criterion}" for i, criterion in enumerate(row["criteria"]))
    user_prompt = (
        f"Question:\n{row['question']}\n\n"
        f"Desired answer:\n{row['desired_answer']}\n\n"
        f"Student answer:\n{row['student_answer']}\n\n"
        f"Criteria:\n{criteria_text}"
    )
    target = json.dumps([int(x) for x in row["criteria_score"]], separators=(",", ":"))
    return {
        "messages": [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [{"type": "text", "text": user_prompt}]},
            {"role": "assistant", "content": [{"type": "text", "text": target}]},
        ]
    }

examples = [make_example(row) for _, row in df.iterrows()]
group_ids = df["id"].drop_duplicates().sample(frac=1, random_state=SEED).tolist()
n_train_groups = int(len(group_ids) * 0.80)
n_valid_groups = int(len(group_ids) * 0.10)
split_group_ids = {
    "train": set(group_ids[:n_train_groups]),
    "valid": set(group_ids[n_train_groups:n_train_groups + n_valid_groups]),
    "test": set(group_ids[n_train_groups + n_valid_groups:]),
}
split_indices = {
    name: df.index[df["id"].isin(ids)].tolist()
    for name, ids in split_group_ids.items()
}
assert not (split_group_ids["train"] & split_group_ids["valid"] | split_group_ids["train"] & split_group_ids["test"] | split_group_ids["valid"] & split_group_ids["test"])
splits = {name: [make_example(df.loc[i]) for i in indices] for name, indices in split_indices.items()}
print({name: {"groups": len(split_group_ids[name]), "rows": len(rows)} for name, rows in splits.items()})
print(json.dumps(splits["train"][0], indent=2, ensure_ascii=False))

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
for split_name, rows in splits.items():
    path = DATA_DIR / f"{split_name}.jsonl"
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"Wrote {len(rows):,} examples to {path}")

## 6. Verify sequence lengths

This downloads tokenizer files only. The cell fails rather than silently truncating training examples. If your CSV changes and the maximum exceeds 512, increase the cap cautiously because activation memory grows with sequence length.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def formatted_token_count(example):
    kwargs = {"tokenize": True, "add_generation_prompt": False}
    try:
        ids = tokenizer.apply_chat_template(example["messages"], enable_thinking=False, **kwargs)
    except TypeError:
        ids = tokenizer.apply_chat_template(example["messages"], **kwargs)
    input_ids = ids["input_ids"] if hasattr(ids, "keys") else ids
    return len(input_ids)

lengths = [formatted_token_count(example) for example in examples]
print(pd.Series(lengths).describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
if max(lengths) > MAX_SEQ_LENGTH:
    raise ValueError(
        f"Longest example is {max(lengths)} tokens, above MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}. "
        "Increase the cap or shorten the examples; do not train with silent truncation."
    )
print("All examples fit. Longest sequence:", max(lengths))

## 7. Write the Qwen3.5-9B MLX-LM configuration

Adapters target both Qwen3.5-9B attention families: Q/K/V/O projections in full-attention layers and QKV/Z/output projections in linear-attention layers. MLP adapters are omitted to control memory and training time. On the 48 GB profile, rank 16 with MLX scale 2.0 corresponds to PEFT alpha 32.

The 48 GB profile clears the allocator cache when cached allocations exceed 1 GiB, allowing reuse between smaller steps. This threshold is not a total-memory limit. Gradient checkpointing remains enabled.


In [ ]:
if any(len(splits[name]) < BATCH_SIZE for name in ("train", "valid")):
    raise ValueError("Training and validation splits must each contain at least one full micro-batch.")
# MLX-LM emits only full batches, discarding any incomplete final batch.
train_micro_batches_per_epoch = len(splits["train"]) // BATCH_SIZE
REQUESTED_ITERS = train_micro_batches_per_epoch * NUM_EPOCHS
# MLX-LM does not flush a partial gradient accumulation at the end.
TOTAL_ITERS = math.ceil(REQUESTED_ITERS / GRAD_ACCUMULATION_STEPS) * GRAD_ACCUMULATION_STEPS
TOTAL_OPTIMIZER_STEPS = TOTAL_ITERS // GRAD_ACCUMULATION_STEPS
WARMUP_UPDATES = max(2, math.ceil(TOTAL_OPTIMIZER_STEPS * 0.05))
# MLX schedules are zero-indexed and switch at warmup + 1.
WARMUP_STEPS = WARMUP_UPDATES - 1
DECAY_UPDATES = TOTAL_OPTIMIZER_STEPS - WARMUP_UPDATES
DECAY_STEPS = max(1, DECAY_UPDATES - 1)
def align_to_optimizer_update(iterations):
    return max(GRAD_ACCUMULATION_STEPS, round(iterations / GRAD_ACCUMULATION_STEPS) * GRAD_ACCUMULATION_STEPS)
STEPS_PER_EVAL = align_to_optimizer_update(max(50, train_micro_batches_per_epoch // 4))
SAVE_EVERY = align_to_optimizer_update(max(100, train_micro_batches_per_epoch // 2))

config = {
    "model": MODEL_ID,
    "train": True,
    "fine_tune_type": "lora",
    "optimizer": "adamw",
    "optimizer_config": {"adamw": {"weight_decay": 0.01}},
    "data": str(DATA_DIR),
    "seed": SEED,
    "num_layers": PROFILE["num_layers"],
    "batch_size": BATCH_SIZE,
    "grad_accumulation_steps": GRAD_ACCUMULATION_STEPS,
    "iters": TOTAL_ITERS,
    "val_batches": -1,  # Compare checkpoints on all full validation batches.
    "learning_rate": LEARNING_RATE,
    "steps_per_report": 10,
    "steps_per_eval": STEPS_PER_EVAL,
    "save_every": SAVE_EVERY,
    "adapter_path": str(OUTPUT_DIR),
    "max_seq_length": MAX_SEQ_LENGTH,
    "grad_checkpoint": PROFILE["grad_checkpoint"],
    # Reuse up to 1 GiB of cached allocations on the M5 Pro 48 GB profile.
    "clear_cache_threshold": PROFILE["clear_cache_threshold"],
    "mask_prompt": True,
    "lora_parameters": {
        "keys": [
            "self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "self_attn.o_proj",
            "linear_attn.in_proj_qkv", "linear_attn.in_proj_z", "linear_attn.out_proj",
        ],
        "rank": PROFILE["rank"],
        # MLX applies scale directly; alpha = 2 * rank (32 for rank 16).
        "scale": 2.0,
        "dropout": LORA_DROPOUT,
    },
    "lr_schedule": {
        "name": "cosine_decay",
        "warmup": WARMUP_STEPS,
        "warmup_init": 1e-7,
        "arguments": [LEARNING_RATE, DECAY_STEPS, 5e-7],
    },
}

CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
print(CONFIG_PATH.read_text())
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUMULATION_STEPS}")
print(f"Iterations: {TOTAL_ITERS:,} (~{TOTAL_ITERS / train_micro_batches_per_epoch:.3f} passes over training data)")
print(f"Optimizer updates: {TOTAL_OPTIMIZER_STEPS:,}; warmup updates: {WARMUP_UPDATES:,}")

## 8. Train Qwen3.5-9B

First set `RUN_TRAINING = True`. Keep the M5 Pro Mac connected to power and close large applications. The training command uses `caffeinate` to prevent idle system sleep. The model downloads on the first run. A validation callback saves the lowest-loss adapter immediately to `OUTPUT_DIR/best/`, including its adapter configuration and `best_checkpoint.json`. Selection uses all full validation batches (`val_batches: -1`), so checks take longer but compare the same examples; MLX omits an incomplete final batch. The initial validation is eligible as a baseline. Equal or non-finite losses do not replace the best checkpoint. Periodic recovery checkpoints remain in the main adapter directory. After training, the actual final weights are also evaluated, and the best weights replace the default `OUTPUT_DIR/adapters.safetensors`. Inference loads `best/` explicitly, which also preserves the best checkpoint if training is interrupted after a successful validation. Check reported peak memory and Activity Monitor during the first training reports.

If memory pressure is high or Metal reports an out-of-memory error, first set `BATCH_SIZE = 1` and `GRAD_ACCUMULATION_STEPS = 8` in the experiment configuration, preserving the effective batch size. If needed, reduce `PROFILE["num_layers"]` to 8 and `PROFILE["rank"]` to 8, and set `PROFILE["clear_cache_threshold"] = 0`. Regenerate the YAML before rerunning training. Start a fresh adapter directory if you change the layer count or rank.

If a long Qwen3.5-9B run reports a Metal resource-limit error, restart Python and reload compatible adapter weights by adding `resume_adapter_file` to the YAML configuration. Adapter loading does not restore optimizer or scheduler state; set the remaining iteration count explicitly.


In [ ]:
RUN_TRAINING = True  # Set True to start Qwen3.5-9B training after reviewing the configuration.

# Run in a separate process so training memory is released before inference.
TRAINING_SCRIPT = OUTPUT_DIR / "train_best.py"
training_script_source = r'''
import json
import math
import shutil
import sys
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import mlx.core as mx
import yaml
from mlx.utils import tree_flatten
from mlx_lm import load
from mlx_lm.lora import CONFIG_DEFAULTS, train_model
from mlx_lm.tuner.callbacks import TrainingCallback
from mlx_lm.tuner.datasets import CacheDataset, load_dataset
from mlx_lm.tuner.trainer import evaluate


class BestAdapter(TrainingCallback):
    def __init__(self, model, output_dir):
        self.model = model
        self.output_dir = Path(output_dir)
        self.best_dir = self.output_dir / "best"
        self.best_dir.mkdir(parents=True, exist_ok=True)
        self.best_loss = math.inf
        self.best_info = None

    def on_val_loss_report(self, info):
        loss = float(info["val_loss"])
        if not math.isfinite(loss) or loss >= self.best_loss:
            return
        # Save the live adapter that produced this loss, before the next update.
        weights = dict(tree_flatten(self.model.trainable_parameters()))
        temporary = self.best_dir / "pending.safetensors"
        mx.save_safetensors(str(temporary), weights)
        temporary.replace(self.best_dir / "adapters.safetensors")
        shutil.copy2(self.output_dir / "adapter_config.json", self.best_dir / "adapter_config.json")
        self.best_loss = loss
        self.best_info = {
            "val_loss": loss,
            "completed_micro_batches": int(info["iteration"]),
        }
        (self.best_dir / "best_checkpoint.json").write_text(
            json.dumps(self.best_info, indent=2) + "\n", encoding="utf-8"
        )
        print(f"Saved best adapter: {self.best_info}", flush=True)

    def promote(self):
        if self.best_info is None:
            raise RuntimeError("No finite validation loss was recorded; no best adapter selected.")
        # MLX writes final weights at training exit. Replace that default with the best.
        temporary = self.output_dir / "selected.safetensors"
        shutil.copy2(self.best_dir / "adapters.safetensors", temporary)
        temporary.replace(self.output_dir / "adapters.safetensors")
        print(f"Default adapter now contains the best validation weights: {self.best_info}", flush=True)


def main():
    values = dict(CONFIG_DEFAULTS)
    values.update(yaml.safe_load(Path(sys.argv[1]).read_text()))
    args = SimpleNamespace(**values)
    np.random.seed(args.seed)
    model, tokenizer = load(args.model)
    train_set, valid_set, _ = load_dataset(args, tokenizer)
    if len(valid_set) < args.batch_size:
        raise ValueError("Best-weight selection requires at least one full validation batch.")
    callback = BestAdapter(model, args.adapter_path)
    train_model(args, model, train_set, valid_set, training_callback=callback)
    # MLX's final in-loop validation runs before its last training update.
    # Also compare the actual final weights using the same validation batches.
    final_loss = evaluate(
        model=model, dataset=CacheDataset(valid_set), batch_size=args.batch_size,
        num_batches=-1, max_seq_length=args.max_seq_length,
    )
    callback.on_val_loss_report({"val_loss": final_loss, "iteration": args.iters})
    callback.promote()


if __name__ == "__main__":
    main()
'''

command = ["caffeinate", "-dimsu", sys.executable, str(TRAINING_SCRIPT), str(CONFIG_PATH)]
print("Command:", " ".join(command))
if RUN_TRAINING:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    TRAINING_SCRIPT.write_text(training_script_source, encoding="utf-8")
    subprocess.run(command, check=True)
else:
    print("Training not started. Set RUN_TRAINING=True and rerun this cell.")


## 9. Load the best adapter and run deterministic inference

Run this after at least one successful validation has saved `best/adapters.safetensors`. The model and the adapter with the lowest validation loss are loaded once and reused for inference and held-out evaluation. `best/best_checkpoint.json` records the loss and number of completed micro-batches; zero denotes the initial baseline validation.


In [ ]:
from mlx_lm import generate, load
from mlx_lm.sample_utils import make_sampler

BEST_ADAPTER_DIR = OUTPUT_DIR / "best"
adapter_file = BEST_ADAPTER_DIR / "adapters.safetensors"
if not adapter_file.exists():
    raise FileNotFoundError(f"Train first; adapter not found at {adapter_file}")

model, mlx_tokenizer = load(MODEL_ID, adapter_path=str(BEST_ADAPTER_DIR))
greedy_sampler = make_sampler(temp=0.0)

def grading_messages(question, desired_answer, student_answer, criteria):
    criteria_text = "\n".join(f"{i + 1}. {criterion}" for i, criterion in enumerate(criteria))
    user = (
        f"Question:\n{question}\n\nDesired answer:\n{desired_answer}\n\n"
        f"Student answer:\n{student_answer}\n\nCriteria:\n{criteria_text}"
    )
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "text", "text": user}]},
    ]

def predict_scores(question, desired_answer, student_answer, criteria):
    messages = grading_messages(question, desired_answer, student_answer, criteria)
    kwargs = {"add_generation_prompt": True}
    try:
        prompt = mlx_tokenizer.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:
        prompt = mlx_tokenizer.apply_chat_template(messages, **kwargs)
    raw = generate(
        model, mlx_tokenizer, prompt=prompt, max_tokens=max(32, len(criteria) * 6),
        sampler=greedy_sampler, verbose=False,
    ).strip()
    candidates = re.findall(r"\[\s*[01](?:\s*,\s*[01])*\s*\]", raw)
    scores = json.loads(candidates[-1]) if candidates else None
    valid = isinstance(scores, list) and len(scores) == len(criteria) and all(x in (0, 1) for x in scores)
    return {"valid": valid, "scores": scores if valid else None, "raw_output": raw}

predict_scores(
    question="Explain why HTTPS is more secure than HTTP.",
    desired_answer="HTTPS encrypts network traffic using TLS.",
    student_answer="HTTPS protects transmitted data using encryption.",
    criteria=["Mentions encryption", "Mentions TLS", "Claims HTTPS sends plaintext"],
)

## 10. Evaluate held-out examples

The default limit keeps evaluation practical on a laptop. Set `EVAL_LIMIT = None` for the complete test set. Invalid model responses count as incorrect for exact-match accuracy.

In [ ]:
EVAL_LIMIT = 50
test_indices = split_indices["test"] if EVAL_LIMIT is None else split_indices["test"][:EVAL_LIMIT]
records = []
flat_true, flat_pred = [], []

for row_index in test_indices:
    row = df.loc[row_index]
    truth = [int(x) for x in row["criteria_score"]]
    prediction = predict_scores(
        str(row["question"]), str(row["desired_answer"]),
        str(row["student_answer"]), row["criteria"],
    )
    exact = bool(prediction["valid"] and prediction["scores"] == truth)
    if prediction["valid"]:
        flat_true.extend(truth)
        flat_pred.extend(prediction["scores"])
    records.append({
        "row_index": int(row_index), "truth": truth,
        "prediction": prediction["scores"] if prediction["valid"] else prediction["raw_output"],
        "valid": prediction["valid"], "exact_match": exact,
    })

evaluation_df = pd.DataFrame(records)
tp = sum(t == 1 and p == 1 for t, p in zip(flat_true, flat_pred))
fp = sum(t == 0 and p == 1 for t, p in zip(flat_true, flat_pred))
fn = sum(t == 1 and p == 0 for t, p in zip(flat_true, flat_pred))
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
metrics = {
    "examples": len(evaluation_df),
    "invalid_output_rate": 1 - evaluation_df["valid"].mean() if len(evaluation_df) else 0.0,
    "exact_match_accuracy": evaluation_df["exact_match"].mean() if len(evaluation_df) else 0.0,
    "per_criterion_accuracy_on_valid": sum(t == p for t, p in zip(flat_true, flat_pred)) / len(flat_true) if flat_true else 0.0,
    "precision_on_valid": precision,
    "recall_on_valid": recall,
    "f1_on_valid": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
}
display(pd.Series(metrics, name="fine_tuned_adapter"))
evaluation_df.head(20)

## Operational notes

- Activity Monitor's Memory Pressure is more useful than free-memory figures on macOS. Keep it green.
- The M5 Pro 48 GB starting profile uses 16 adapter layers, rank 16, batch size 2, four accumulation steps, and gradient checkpointing. Confirm memory use locally before increasing sequence length or batch size.
- A 32 GB Mac uses the 8-layer/rank-8 fallback with batch size 1 and eight accumulation steps.
- For repeatable Qwen3.5-9B comparisons, retain the generated JSONL split files and YAML configuration with the adapter.
- Evaluate the Qwen3.5-9B adapter before fusing; keeping it separate is smaller and makes experiments easier to compare.
- The adapter is framework-specific. Export or fuse through MLX-LM if the final deployment format requires merged weights.

References: [Qwen3.5-9B checkpoint files and size](https://huggingface.co/mlx-community/Qwen3.5-9B-4bit/tree/main), [MLX-LM training implementation](https://github.com/ml-explore/mlx-lm/blob/main/mlx_lm/tuner/trainer.py).

Best-checkpoint selection uses the [MLX-LM validation callback](https://github.com/ml-explore/mlx-lm/blob/main/mlx_lm/tuner/callbacks.py) to save the weights at the time their validation loss is measured.
